# Behavioral-cloning baseline training

Standalone Colab training for the existing CNN → spatial-soft-argmax → MLP policy. Uses the same 20,000-episode Lift HDF5 as primary training, train-only proprioception normalization, bounded OSC actions, spawn-safe parallel loading, warmup plus cosine learning-rate decay, and validation-selected checkpoints.


In [ ]:
%pip install -q h5py google-api-python-client


In [ ]:
import json
import shutil
from pathlib import Path

import google.auth
import h5py
from google.colab import auth, drive
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from tqdm.auto import tqdm

# The mount is still used for checkpoints, but the large dataset is not
# read through DriveFS because DriveFS can cache a second 37+ GiB copy.
drive.mount('/content/drive')
auth.authenticate_user()
credentials, _ = google.auth.default()
drive_api = build('drive', 'v3', credentials=credentials, cache_discovery=False)

def find_drive_child(parent_id, name, mime_type=None):
    query = f"'{parent_id}' in parents and name = '{name}' and trashed = false"
    if mime_type is not None:
        query += f" and mimeType = '{mime_type}'"
    result = drive_api.files().list(
        q=query, spaces='drive', orderBy='modifiedTime desc', pageSize=10,
        fields='files(id,name,size,mimeType,modifiedTime)',
    ).execute()
    files = result.get('files', [])
    return files[0] if files else None

root_id = drive_api.files().get(fileId='root', fields='id').execute()['id']
primary_folder = find_drive_child(
    root_id, 'primary', 'application/vnd.google-apps.folder'
)
if primary_folder is None:
    raise FileNotFoundError('Could not find MyDrive/primary through the Drive API')

drive_file = None
candidate_groups = (
    ('demo_rechunked.hdf5', 'demos_rechunked.hdf5'),
    ('demo.hdf5', 'demos.hdf5'),
)
for candidate_names in candidate_groups:
    matches = [find_drive_child(primary_folder['id'], name) for name in candidate_names]
    matches = [match for match in matches if match is not None]
    if matches:
        drive_file = max(matches, key=lambda match: match['modifiedTime'])
        break
if drive_file is None:
    raise FileNotFoundError(
        'Could not find a rechunked or standard demo HDF5 in MyDrive/primary'
    )
source_size = int(drive_file['size'])

# Stage exactly one HDF5 on local fast storage for training reads.
local_dir = Path('/content/data/robot/mimicgen')
local_dir.mkdir(parents=True, exist_ok=True)
local_h5 = local_dir / 'demo.hdf5'
partial_h5 = local_dir / 'demo.hdf5.partial'

# Remove only known duplicate / interrupted local copies. Never touch Drive.
for duplicate in (local_dir / 'demo_rechunked.hdf5', local_dir / 'demos_rechunked.hdf5', local_dir / 'demos.hdf5', partial_h5):
    if duplicate.exists():
        duplicate.unlink()
        print(f'removed duplicate local file: {duplicate}')

if local_h5.exists() and local_h5.stat().st_size == source_size:
    print(f'reusing complete local copy: {local_h5}')
else:
    if local_h5.exists():
        local_h5.unlink()
        print(f'removed incomplete or outdated local copy: {local_h5}')

    free_bytes = shutil.disk_usage('/content').free
    required_bytes = source_size + 5 * 1024**3
    if free_bytes < required_bytes:
        raise RuntimeError(
            f'Not enough local disk space: {free_bytes / 1024**3:.1f} GiB free, '
            f'but at least {required_bytes / 1024**3:.1f} GiB is required. '
            'Use Runtime > Disconnect and delete runtime, then rerun.'
        )

    print(
        f"streaming MyDrive/primary/{drive_file['name']} → {local_h5} "
        f'({source_size / 1024**3:.2f} GiB; no DriveFS dataset cache)'
    )
    request = drive_api.files().get_media(fileId=drive_file['id'])
    with partial_h5.open('wb') as destination:
        downloader = MediaIoBaseDownload(
            destination, request, chunksize=64 * 1024**2
        )
        done = False
        with tqdm(
            total=source_size, unit='B', unit_scale=True, unit_divisor=1024,
            desc='dataset download', dynamic_ncols=True,
        ) as progress:
            while not done:
                _, done = downloader.next_chunk(num_retries=5)
                progress.update(destination.tell() - progress.n)
    if partial_h5.stat().st_size != source_size:
        partial_h5.unlink(missing_ok=True)
        raise IOError('Local dataset copy has the wrong size; partial copy removed')
    partial_h5.replace(local_h5)

# Fail here, before training, if the staged file is not the expected dataset.
with h5py.File(local_h5, 'r') as file:
    if 'data' not in file or 'mask/train' not in file or 'mask/valid' not in file:
        raise ValueError(f'Invalid MimicGen HDF5 structure: {local_h5}')
    env_args = file['data'].attrs.get('env_args')
    if isinstance(env_args, bytes):
        env_args = env_args.decode('utf-8')
    env_name = json.loads(env_args).get('env_name') if env_args else None
    if env_name != 'Lift_D0':
        raise ValueError(f'Expected Lift_D0 training data, but found {env_name!r} in {local_h5}')
    train_episodes = len(file['mask/train'])
    valid_episodes = len(file['mask/valid'])

data_dir = Path('/content/data')
free_after = shutil.disk_usage('/content').free / 1024**3
print(f'dataset ready: {local_h5} ({local_h5.stat().st_size / 1024**3:.2f} GiB)')
print(f'environment: {env_name}; episodes: {train_episodes} train / {valid_episodes} validation; local disk free: {free_after:.1f} GiB')


In [ ]:
%%writefile /content/baseline_dataset_runtime.py
import re
from bisect import bisect_right
from pathlib import Path

import h5py
import torch
from torch.utils.data import Dataset


def _demo_number(name: str) -> int:
    return int(re.search(r"(\d+)", name).group(1))


def _decode_demo_name(raw) -> str:
    return raw.decode("utf-8") if isinstance(raw, bytes) else str(raw)


def build_robot_state(pos, quat, grip):
    quat = quat / quat.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    return torch.cat((pos, quat, grip), dim=-1)


def normalize_robot_state(state, state_mean, state_std):
    return (state - state_mean) / state_std


def compute_train_state_stats(h5_path: Path):
    """Compute 9-D proprioception statistics from training episodes only."""
    count = 0
    total = torch.zeros(9, dtype=torch.float64)
    total_sq = torch.zeros(9, dtype=torch.float64)
    with h5py.File(h5_path, "r") as file:
        episodes = [_decode_demo_name(x) for x in file["mask/train"][:]]
        for name in episodes:
            obs = file[f"data/{name}/obs"]
            state = build_robot_state(
                torch.from_numpy(obs["robot0_eef_pos"][:]).to(torch.float64),
                torch.from_numpy(obs["robot0_eef_quat"][:]).to(torch.float64),
                torch.from_numpy(obs["robot0_gripper_qpos"][:]).to(torch.float64),
            )
            count += state.shape[0]
            total += state.sum(dim=0)
            total_sq += state.square().sum(dim=0)
    if count == 0:
        raise ValueError(f"No training observations found in {h5_path}")
    mean = total / count
    variance = (total_sq / count) - mean.square()
    std = variance.clamp_min(1e-12).sqrt().clamp_min(1e-6)
    return mean.float(), std.float()


class RobotBaselineDataset(Dataset):
    """One observation/action pair per primitive simulator step."""
    def __init__(self, data_dir: Path, val=False, state_mean=None, state_std=None):
        super().__init__()
        self.h5_path = Path(data_dir) / "robot" / "mimicgen" / "demo.hdf5"
        if not self.h5_path.exists():
            raise FileNotFoundError(f"Dataset not found: {self.h5_path}")
        if state_mean is None or state_std is None:
            state_mean, state_std = compute_train_state_stats(self.h5_path)
        self.state_mean = torch.as_tensor(state_mean, dtype=torch.float32)
        self.state_std = torch.as_tensor(state_std, dtype=torch.float32)
        if self.state_mean.shape != (9,) or self.state_std.shape != (9,):
            raise ValueError("state_mean and state_std must both have shape (9,)")

        mask_key = "valid" if val else "train"
        self.episodes = []
        self.cumulative_frames = []
        total_frames = 0
        with h5py.File(self.h5_path, "r") as file:
            names = [_decode_demo_name(x) for x in file[f"mask/{mask_key}"][:]]
            for name in sorted(names, key=_demo_number):
                length = int(file[f"data/{name}"].attrs.get(
                    "num_samples", file[f"data/{name}/actions"].shape[0]
                ))
                if length > 0:
                    total_frames += length
                    self.episodes.append(name)
                    self.cumulative_frames.append(total_frames)
        self.total_len = total_frames
        self._h5 = None

    def __len__(self):
        return self.total_len

    def _get_h5(self):
        if self._h5 is None:
            self._h5 = h5py.File(self.h5_path, "r")
        return self._h5

    def _resolve_index(self, index):
        if index < 0:
            index += self.total_len
        if index < 0 or index >= self.total_len:
            raise IndexError(index)
        episode_index = bisect_right(self.cumulative_frames, index)
        episode_start = 0 if episode_index == 0 else self.cumulative_frames[episode_index - 1]
        return self.episodes[episode_index], index - episode_start

    def __getstate__(self):
        state = self.__dict__.copy()
        state["_h5"] = None
        return state

    def close(self):
        if getattr(self, "_h5", None) is not None:
            self._h5.close()
            self._h5 = None

    def __del__(self):
        self.close()

    def __getitem__(self, index):
        name, frame = self._resolve_index(index)
        episode = self._get_h5()[f"data/{name}"]
        obs = episode["obs"]
        above = torch.from_numpy(obs["agentview_image"][frame]).permute(2, 0, 1)
        wrist = torch.from_numpy(obs["robot0_eye_in_hand_image"][frame]).permute(2, 0, 1)
        state = build_robot_state(
            torch.from_numpy(obs["robot0_eef_pos"][frame]).float(),
            torch.from_numpy(obs["robot0_eef_quat"][frame]).float(),
            torch.from_numpy(obs["robot0_gripper_qpos"][frame]).float(),
        )
        state = normalize_robot_state(state, self.state_mean, self.state_std)
        action = torch.from_numpy(episode["actions"][frame]).float()
        return above, wrist, state, action


In [ ]:
import importlib
import sys
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

module_dir = Path('/content')
if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))
importlib.invalidate_caches()
sys.modules.pop('baseline_dataset_runtime', None)
_dataset_module = importlib.import_module('baseline_dataset_runtime')
RobotBaselineDataset = _dataset_module.RobotBaselineDataset
compute_train_state_stats = _dataset_module.compute_train_state_stats
print('spawn-safe dataset module ready: /content/baseline_dataset_runtime.py')


class SpatialSoftArgmax(nn.Module):
    """Per-channel spatial softmax → expected (x, y) in [-1, 1]."""
    def forward(self, x):
        batch, channels, height, width = x.shape
        weights = F.softmax(x.reshape(batch, channels, -1), dim=-1).reshape(
            batch, channels, height, width
        )
        ys = torch.linspace(-1.0, 1.0, height, device=x.device, dtype=x.dtype)
        xs = torch.linspace(-1.0, 1.0, width, device=x.device, dtype=x.dtype)
        expected_y = (weights.sum(dim=-1) * ys.view(1, 1, height)).sum(dim=-1)
        expected_x = (weights.sum(dim=-2) * xs.view(1, 1, width)).sum(dim=-1)
        return torch.stack((expected_x, expected_y), dim=-1).reshape(batch, channels * 2)


class BaselinePolicy(nn.Module):
    """Original 8.2M-parameter CNN + spatial-softmax + MLP BC policy."""
    def __init__(self, d_state=9, d_action=7, hidden=1536):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(6, 128, kernel_size=7, stride=2, padding=3),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=5, stride=2, padding=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 384, kernel_size=5, stride=2, padding=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 384, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
        )
        self.spatial_softargmax = SpatialSoftArgmax()
        self.action_head = nn.Sequential(
            nn.Linear(384 * 2 + d_state, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, d_action),
        )

    def forward(self, above, wrist, state):
        features = self.spatial_softargmax(self.cnn(torch.cat((above, wrist), dim=1)))
        raw = self.action_head(torch.cat((features, state), dim=-1))
        continuous = torch.tanh(raw[:, :6])
        return torch.cat((continuous, raw[:, 6:7]), dim=-1)


In [ ]:
import contextlib
import inspect
import math
import re
import shutil
import tempfile
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm

ACTIVE_ACTION_DIMS = tuple(range(6))
ACTION_NAMES = ('x', 'y', 'z', 'roll', 'pitch', 'yaw')
ARCHITECTURE = 'cnn_spatial_softargmax_bounded_all_motion_cosine_v4'


def baseline_loss(prediction, target, gripper_coeff):
    motion = nn.functional.mse_loss(
        prediction[:, ACTIVE_ACTION_DIMS], target[:, ACTIVE_ACTION_DIMS]
    )
    gripper_target = (target[:, 6] > 0).float()
    gripper = nn.functional.binary_cross_entropy_with_logits(
        prediction[:, 6], gripper_target
    )
    return motion + gripper_coeff * gripper, motion, gripper


def scheduled_learning_rate(step, peak_lr, min_lr, warmup_steps, total_steps):
    if warmup_steps > 0 and step < warmup_steps:
        return peak_lr * (step + 1) / warmup_steps
    if total_steps <= warmup_steps + 1:
        return min_lr
    progress = min(max((step - warmup_steps) / (total_steps - warmup_steps - 1), 0.0), 1.0)
    return min_lr + 0.5 * (peak_lr - min_lr) * (1.0 + math.cos(math.pi * progress))


def loader_options(num_workers, persistent_workers, pin_memory, prefetch_factor=2):
    options = {
        'num_workers': num_workers,
        'persistent_workers': persistent_workers and num_workers > 0,
        'pin_memory': pin_memory,
    }
    if num_workers > 0:
        options.update({
            'prefetch_factor': prefetch_factor,
            'multiprocessing_context': 'spawn',
            'timeout': 120,
        })
    if 'in_order' in inspect.signature(DataLoader).parameters:
        options['in_order'] = True
    return options


def atomic_save(payload, destination: Path):
    destination.parent.mkdir(parents=True, exist_ok=True)
    pending = destination.with_suffix(destination.suffix + '.tmp')
    pending.unlink(missing_ok=True)
    with tempfile.TemporaryDirectory() as temporary:
        local = Path(temporary) / destination.name
        torch.save(payload, local)
        shutil.copyfile(local, pending)
    pending.replace(destination)


def latest_baseline_checkpoint(save_dir: Path):
    checkpoints = []
    for path in save_dir.glob('epoch_*_baseline.pth'):
        match = re.fullmatch(r'epoch_(\d+)_baseline\.pth', path.name)
        if match:
            checkpoints.append((int(match.group(1)), path))
    return max(checkpoints, default=(None, None), key=lambda item: item[0])


def prepare_output_dirs(save_dir: Path, log_dir: Path, resume: bool):
    save_dir.mkdir(parents=True, exist_ok=True)
    if not resume:
        stamp = time.strftime('%Y%m%d_%H%M%S')
        old_checkpoints = list(save_dir.glob('epoch_*_baseline.pth'))
        old_checkpoints += [
            path for path in (save_dir / 'baseline_latest.pth', save_dir / 'best_baseline.pth')
            if path.exists()
        ]
        if old_checkpoints:
            archive = save_dir / 'archive' / stamp
            archive.mkdir(parents=True, exist_ok=False)
            for path in old_checkpoints:
                shutil.move(str(path), str(archive / path.name))
            print(f'archived previous checkpoints → {archive}')
        if log_dir.exists() and any(log_dir.iterdir()):
            archive = log_dir.parent / 'log_archives' / stamp
            archive.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(log_dir), str(archive))
            print(f'archived previous logs → {archive}')
    log_dir.mkdir(parents=True, exist_ok=True)


def train(device, data_dir, log_dir, save_dir, lr, seed, gripper_coeff,
          epochs, batch_size, num_workers, persistent_workers, pin_memory,
          optimizer_betas, optimizer_weight_decay, gradient_clip_norm=1.0,
          min_lr=1e-5, warmup_steps=2000, compile_model=True, resume=False):
    torch.manual_seed(seed)
    if not 0 < min_lr <= lr:
        raise ValueError(f'min_lr must be in (0, lr], got min_lr={min_lr}, lr={lr}')
    if warmup_steps < 0:
        raise ValueError('warmup_steps cannot be negative')
    on_cuda = str(device).startswith('cuda')
    if on_cuda:
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.set_float32_matmul_precision('high')

    prepare_output_dirs(save_dir, log_dir, resume)
    h5_path = Path(data_dir) / 'robot' / 'mimicgen' / 'demo.hdf5'
    state_mean, state_std = compute_train_state_stats(h5_path)
    print(f'state mean: {state_mean.tolist()}')
    print(f'state std:  {state_std.tolist()}')

    train_dataset = RobotBaselineDataset(data_dir, False, state_mean, state_std)
    val_dataset = RobotBaselineDataset(data_dir, True, state_mean, state_std)
    options = loader_options(num_workers, persistent_workers, pin_memory)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, **options)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, **options)
    schedule_total_steps = epochs * len(train_loader)
    if schedule_total_steps <= warmup_steps:
        raise ValueError('Training must contain more optimizer steps than warmup_steps')
    print(
        f"DataLoader: workers={num_workers}, start_method="
        f"{options.get('multiprocessing_context', 'single-process')}, "
        f"persistent={options['persistent_workers']}"
    )

    policy = BaselinePolicy().to(device)
    n_params = sum(parameter.numel() for parameter in policy.parameters())
    print(f'BaselinePolicy on {device} | params={n_params:,} ({n_params / 1e6:.2f}M)')

    optimizer_kwargs = dict(
        lr=lr, betas=optimizer_betas, weight_decay=optimizer_weight_decay
    )
    if on_cuda and 'fused' in inspect.signature(torch.optim.AdamW).parameters:
        optimizer_kwargs['fused'] = True
    optimizer = torch.optim.AdamW(policy.parameters(), **optimizer_kwargs)

    history = {
        'train_total': [], 'train_motion': [], 'train_gripper': [],
        'train_gripper_accuracy': [], 'train_action_mse': [],
        'train_grad_norm': [], 'train_clip_fraction': [], 'learning_rate': [],
        'val_total': [], 'val_motion': [], 'val_gripper': [],
        'val_gripper_accuracy': [], 'val_action_mse': [],
    }
    start_epoch = 0
    global_step = 0
    best_val_loss = float('inf')
    best_epoch = -1
    resume_payload = None
    if resume:
        latest_epoch, checkpoint_path = latest_baseline_checkpoint(save_dir)
        if checkpoint_path is None:
            raise FileNotFoundError(f'resume=True but no epoch baseline checkpoint exists in {save_dir}')
        resume_payload = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
        if resume_payload.get('architecture') != ARCHITECTURE:
            raise ValueError('Checkpoint uses an older baseline action contract; start with resume=False')
        if 'optimizer_state_dict' not in resume_payload:
            raise ValueError('Checkpoint has no optimizer state and cannot be resumed exactly')
        if not torch.allclose(resume_payload['state_mean'].cpu(), state_mean):
            raise ValueError('Dataset state mean does not match the checkpoint')
        if not torch.allclose(resume_payload['state_std'].cpu(), state_std):
            raise ValueError('Dataset state std does not match the checkpoint')
        policy.load_state_dict(resume_payload['model_state_dict'])
        optimizer.load_state_dict(resume_payload['optimizer_state_dict'])
        history = resume_payload.get('history', history)
        start_epoch = int(resume_payload.get('epoch', latest_epoch)) + 1
        global_step = int(resume_payload.get('global_step', start_epoch * len(train_loader)))
        schedule_total_steps = int(resume_payload.get('schedule_total_steps', schedule_total_steps))
        saved_peak_lr = float(resume_payload.get('peak_lr', lr))
        saved_min_lr = float(resume_payload.get('min_lr', min_lr))
        saved_warmup_steps = int(resume_payload.get('warmup_steps', warmup_steps))
        if not math.isclose(saved_peak_lr, lr) or not math.isclose(saved_min_lr, min_lr) or saved_warmup_steps != warmup_steps:
            raise ValueError('Resume learning-rate schedule does not match the checkpoint')
        best_val_loss = float(resume_payload.get('best_val_loss', float('inf')))
        best_epoch = int(resume_payload.get('best_epoch', -1))

    train_policy = policy
    if compile_model and hasattr(torch, 'compile'):
        try:
            train_policy = torch.compile(policy)
            print('torch.compile enabled')
        except Exception as error:
            print(f'torch.compile unavailable; continuing eagerly: {error}')

    if on_cuda and torch.cuda.is_bf16_supported():
        amp_dtype = torch.bfloat16
        scaler = None
        print('AMP: bf16')
    elif on_cuda:
        amp_dtype = torch.float16
        scaler = torch.amp.GradScaler('cuda')
        print('AMP: fp16')
    else:
        amp_dtype = None
        scaler = None
        print('AMP: off')

    if resume_payload is not None:
        scaler_state = resume_payload.get('scaler_state_dict', {})
        if scaler is not None and scaler_state:
            scaler.load_state_dict(scaler_state)
        if 'torch_rng_state' in resume_payload:
            torch.set_rng_state(resume_payload['torch_rng_state'])
        if on_cuda and resume_payload.get('cuda_rng_state_all') is not None:
            torch.cuda.set_rng_state_all(resume_payload['cuda_rng_state_all'])
        print(f'resumed {checkpoint_path.name}; continuing at epoch {start_epoch}')

    def autocast_context():
        return torch.amp.autocast('cuda', dtype=amp_dtype) if on_cuda else contextlib.nullcontext()

    writer = SummaryWriter(log_dir=log_dir)
    non_blocking = on_cuda and pin_memory
    for epoch in tqdm(range(start_epoch, epochs), desc='epochs'):
        print(f'\nEpoch {epoch}\n------------')
        train_policy.train()
        train_sums = np.zeros(3, dtype=np.float64)
        train_dim_sq = np.zeros(6, dtype=np.float64)
        train_grip_correct = 0
        train_grad_norm_sum = 0.0
        train_clipped_steps = 0
        train_lr_sum = 0.0
        train_examples = 0

        for above, wrist, state, action in tqdm(train_loader, desc=f'train {epoch}', leave=False):
            current_lr = scheduled_learning_rate(
                global_step, lr, min_lr, warmup_steps, schedule_total_steps
            )
            for parameter_group in optimizer.param_groups:
                parameter_group['lr'] = current_lr
            above = above.to(device, non_blocking=non_blocking).float().mul_(1.0 / 255.0)
            wrist = wrist.to(device, non_blocking=non_blocking).float().mul_(1.0 / 255.0)
            state = state.to(device, non_blocking=non_blocking)
            action = action.to(device, non_blocking=non_blocking)
            optimizer.zero_grad(set_to_none=True)
            with autocast_context():
                prediction = train_policy(above, wrist, state)
                loss, motion, gripper = baseline_loss(prediction, action, gripper_coeff)
            if scaler is None:
                loss.backward()
                grad_norm = nn.utils.clip_grad_norm_(
                    policy.parameters(), gradient_clip_norm, error_if_nonfinite=True
                )
                optimizer.step()
            else:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                grad_norm = nn.utils.clip_grad_norm_(
                    policy.parameters(), gradient_clip_norm, error_if_nonfinite=True
                )
                scaler.step(optimizer)
                scaler.update()
            global_step += 1

            batch = action.shape[0]
            grad_norm_value = float(grad_norm.detach().float().item())
            train_sums += batch * np.array([loss.item(), motion.item(), gripper.item()])
            train_dim_sq += (prediction[:, :6].detach().float() - action[:, :6].float()).square().sum(dim=0).cpu().numpy()
            train_grip_correct += ((prediction[:, 6] > 0) == (action[:, 6] > 0)).sum().item()
            train_grad_norm_sum += batch * grad_norm_value
            train_clipped_steps += int(grad_norm_value > gradient_clip_norm)
            train_lr_sum += batch * current_lr
            train_examples += batch

        train_metrics = train_sums / train_examples
        train_dim_mse = train_dim_sq / train_examples
        train_grip_accuracy = train_grip_correct / train_examples
        train_grad_norm = train_grad_norm_sum / train_examples
        train_clip_fraction = train_clipped_steps / len(train_loader)
        train_mean_lr = train_lr_sum / train_examples

        train_policy.eval()
        val_sums = np.zeros(3, dtype=np.float64)
        val_dim_sq = np.zeros(6, dtype=np.float64)
        val_grip_correct = 0
        val_examples = 0
        with torch.inference_mode():
            for above, wrist, state, action in val_loader:
                above = above.to(device, non_blocking=non_blocking).float().mul_(1.0 / 255.0)
                wrist = wrist.to(device, non_blocking=non_blocking).float().mul_(1.0 / 255.0)
                state = state.to(device, non_blocking=non_blocking)
                action = action.to(device, non_blocking=non_blocking)
                with autocast_context():
                    prediction = train_policy(above, wrist, state)
                    loss, motion, gripper = baseline_loss(prediction, action, gripper_coeff)
                batch = action.shape[0]
                val_sums += batch * np.array([loss.item(), motion.item(), gripper.item()])
                val_dim_sq += (prediction[:, :6].float() - action[:, :6].float()).square().sum(dim=0).cpu().numpy()
                val_grip_correct += ((prediction[:, 6] > 0) == (action[:, 6] > 0)).sum().item()
                val_examples += batch

        val_metrics = val_sums / val_examples
        val_dim_mse = val_dim_sq / val_examples
        val_grip_accuracy = val_grip_correct / val_examples
        history['train_total'].append(float(train_metrics[0]))
        history['train_motion'].append(float(train_metrics[1]))
        history['train_gripper'].append(float(train_metrics[2]))
        history['train_gripper_accuracy'].append(float(train_grip_accuracy))
        history['train_action_mse'].append(train_dim_mse.tolist())
        history['train_grad_norm'].append(float(train_grad_norm))
        history['train_clip_fraction'].append(float(train_clip_fraction))
        history['learning_rate'].append(float(train_mean_lr))
        history['val_total'].append(float(val_metrics[0]))
        history['val_motion'].append(float(val_metrics[1]))
        history['val_gripper'].append(float(val_metrics[2]))
        history['val_gripper_accuracy'].append(float(val_grip_accuracy))
        history['val_action_mse'].append(val_dim_mse.tolist())
        writer.add_scalars(
            'loss/total', {'train': train_metrics[0], 'val': val_metrics[0]}, epoch
        )
        writer.add_scalars(
            'loss/motion', {'train': train_metrics[1], 'val': val_metrics[1]}, epoch
        )
        writer.add_scalars(
            'loss/gripper', {'train': train_metrics[2], 'val': val_metrics[2]}, epoch
        )
        writer.add_scalars('accuracy/gripper', {'train': train_grip_accuracy, 'val': val_grip_accuracy}, epoch)
        writer.add_scalar('optimization/learning_rate', train_mean_lr, epoch)
        writer.add_scalar('optimization/gradient_norm', train_grad_norm, epoch)
        writer.add_scalar('optimization/clip_fraction', train_clip_fraction, epoch)
        for index, name in enumerate(ACTION_NAMES):
            writer.add_scalars(
                f'action_mse/{name}',
                {'train': train_dim_mse[index], 'val': val_dim_mse[index]},
                epoch,
            )
        writer.flush()

        improved = val_metrics[0] < best_val_loss
        if improved:
            best_val_loss = float(val_metrics[0])
            best_epoch = epoch
        payload = {
            'model_state_dict': policy.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': {} if scaler is None else scaler.state_dict(),
            'state_mean': state_mean.cpu(),
            'state_std': state_std.cpu(),
            'history': history,
            'epoch': epoch,
            'global_step': global_step,
            'schedule_total_steps': schedule_total_steps,
            'peak_lr': lr,
            'min_lr': min_lr,
            'warmup_steps': warmup_steps,
            'val_loss': float(val_metrics[0]),
            'best_val_loss': best_val_loss,
            'best_epoch': best_epoch,
            'architecture': ARCHITECTURE,
            'torch_rng_state': torch.get_rng_state(),
            'cuda_rng_state_all': torch.cuda.get_rng_state_all() if on_cuda else None,
        }
        epoch_path = save_dir / f'epoch_{epoch}_baseline.pth'
        atomic_save(payload, epoch_path)
        atomic_save(payload, save_dir / 'baseline_latest.pth')
        if improved:
            atomic_save(payload, save_dir / 'best_baseline.pth')

        print(
            f'train total={train_metrics[0]:.5f} (motion={train_metrics[1]:.5f}, '
            f'grip={train_metrics[2]:.5f}, grip_acc={100 * train_grip_accuracy:.1f}%) | '
            f'val total={val_metrics[0]:.5f} (motion={val_metrics[1]:.5f}, '
            f'grip={val_metrics[2]:.5f}, grip_acc={100 * val_grip_accuracy:.1f}%)'
        )
        print(f'best validation: epoch {best_epoch}, loss {best_val_loss:.5f}')
        print(
            f'optimization: mean_lr={train_mean_lr:.2e}, grad_norm={train_grad_norm:.4f}, '
            f'clipped={100 * train_clip_fraction:.1f}%, global_step={global_step}/{schedule_total_steps}'
        )
        print(f'saved → {epoch_path}')

    writer.close()
    train_dataset.close()
    val_dataset.close()
    print(f'\nDone. best epoch: {best_epoch}; logs: {log_dir}; checkpoints: {save_dir}')


In [ ]:
import time
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/baseline')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
save_dir = DRIVE_DIR / 'models'
log_dir = DRIVE_DIR / 'logs'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {device}' + (f' ({torch.cuda.get_device_name(0)})' if device == 'cuda' else ''))

start_time = time.perf_counter()
train(
    device=device,
    data_dir=data_dir,
    log_dir=log_dir,
    save_dir=save_dir,
    lr=1e-4,
    seed=42,
    gripper_coeff=0.1,
    epochs=10,
    batch_size=128,
    num_workers=8,
    persistent_workers=False,
    pin_memory=True,
    optimizer_betas=(0.9, 0.95),
    optimizer_weight_decay=0.01,
    gradient_clip_norm=1.0,
    min_lr=1e-5,
    warmup_steps=2000,
    compile_model=True,
    resume=False,  # set True and increase epochs to continue from the latest epoch checkpoint
)
print(f'training time: {(time.perf_counter() - start_time):.2f}s')
print(f'checkpoints saved to: {save_dir}')
print(f'logs saved to: {log_dir}')


In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/baseline/logs
